In [ ]:
import geopandas as gpd
import rasterio
from rasterio.mask import mask
import numpy as np
import pandas as pd
from tqdm import tqdm

# Load the Shapefile
shapefile_path = f'G:\Hangkai\CONUS Forest Edge Mapping\CONUS shapefile\CONUS_ECOSYSTEM.shp'
shapes = gpd.read_file(shapefile_path)

# Load Raster Data
year = 2021
raster_path = f'G:\Hangkai\CONUS_Forest_Edge_LCMAP\Edge_adjunct_LC\Adjunct_LC_{year}.tif'
raster = rasterio.open(raster_path)

# Ensure CRS consistency
if shapes.crs != raster.crs:
    shapes = shapes.to_crs(raster.crs)

results = []

# Function to clip the raster and count unique values
def clip_and_count_values(raster, geometry, name, na_l1name):
    out_image, out_transform = mask(raster, [geometry], crop=True)
    unique, counts = np.unique(out_image[out_image > 0], return_counts=True)
    result = {'NAME': name, 'NA_L1NAME': na_l1name}  # Start with name columns
    counts_dict = dict(zip(unique, counts))
    result.update(counts_dict)  # Add count results after name columns
    return result

# Iterate over each polygon
for index, row in tqdm(shapes.iterrows(), total=shapes.shape[0]):
    result = clip_and_count_values(raster, row['geometry'], row['NAME'], row['NA_L1NAME'])
    results.append(result)

# Create DataFrame and save to CSV
df = pd.DataFrame(results)
df.to_csv(f'G:\Hangkai\CONUS_Forest_Edge_LCMAP\Edge_adjunct_LC\Ecoregion_Classification\eco_{year}.csv', index=False)
